In [16]:

from ultralytics import YOLO
from tqdm import tqdm
from collections import defaultdict
import cv2
import numpy as np

source = "./videos/video.mp4"
target = "./videos/result_video.mp4"
model_path = "./runs/detect/soccer_training/ball_detector_640/weights/best.pt"

model = YOLO(model_path)

cap = cv2.VideoCapture(source)

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {source}")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(target, fourcc, fps, (width, height))

if not out.isOpened():
    raise RuntimeError(f"Could not create output video: {target}")

REFEREE_COLOR = (0, 0, 255)
BALL_COLOR = (0, 165, 255)

BALL_CONFIDENCE = 0.03
REFEREE_CONFIDENCE = 0.20

BALL_MAX_MISSED_FRAMES = 10
REFEREE_MAX_MISSED_FRAMES = 10

BALL_BASE_MAX_DISTANCE = 150
REFEREE_BASE_MAX_DISTANCE = 120

BALL_SMOOTHING = 0.65
REFEREE_SMOOTHING = 0.65

last_ball_center = None
last_ball_box = None
previous_ball_center = None
missed_ball_frames = 0
last_ball_confidence = 0

referee_tracks = {}
next_referee_id = 0


def predict_position(track):
    velocity = np.array(track["center"]) - np.array(track["previous"])
    return np.array(track["center"]) + velocity * (track["missed"] + 1)


for _ in tqdm(range(total_frames)):

    ret, frame = cap.read()

    if not ret:
        break

    result = model.predict(
        frame,
        conf=BALL_CONFIDENCE,
        iou=0.5,
        imgsz=1280,
        max_det=300,
        verbose=False
    )[0]

    ball_candidates = []
    referee_candidates = []

    for box in result.boxes:

        confidence = float(box.conf[0])
        class_id = int(box.cls[0])
        class_name = model.names[class_id].lower()

        x1, y1, x2, y2 = map(int, box.xyxy[0])

        center = (
            (x1 + x2) // 2,
            (y1 + y2) // 2
        )

        if "ball" in class_name:

            ball_candidates.append({
                "box": (x1, y1, x2, y2),
                "center": center,
                "confidence": confidence
            })

        elif "referee" in class_name and confidence >= REFEREE_CONFIDENCE:

            referee_candidates.append({
                "box": (x1, y1, x2, y2),
                "center": center,
                "confidence": confidence
            })

    selected_ball = None

    if ball_candidates:

        if last_ball_center is None or missed_ball_frames > BALL_MAX_MISSED_FRAMES:

            selected_ball = max(
                ball_candidates,
                key=lambda item: item["confidence"]
            )

        else:

            if previous_ball_center is not None:

                velocity = (
                    np.array(last_ball_center) -
                    np.array(previous_ball_center)
                )

                predicted_center = (
                    np.array(last_ball_center) +
                    velocity * (missed_ball_frames + 1)
                )

                max_distance = BALL_BASE_MAX_DISTANCE + (
                    np.linalg.norm(velocity) * (missed_ball_frames + 1)
                )

            else:

                predicted_center = np.array(last_ball_center)
                max_distance = BALL_BASE_MAX_DISTANCE

            valid_candidates = []

            for candidate in ball_candidates:

                distance = np.linalg.norm(
                    np.array(candidate["center"]) - predicted_center
                )

                if distance <= max_distance:
                    valid_candidates.append(candidate)

            if valid_candidates:

                selected_ball = min(
                    valid_candidates,
                    key=lambda item: np.linalg.norm(
                        np.array(item["center"]) - predicted_center
                    )
                )

    if selected_ball is not None:

        detected_center = np.array(selected_ball["center"])

        if last_ball_center is not None and missed_ball_frames == 0:

            smoothed_center = (
                BALL_SMOOTHING * detected_center +
                (1 - BALL_SMOOTHING) * np.array(last_ball_center)
            )

            smoothed_center = tuple(map(int, smoothed_center))

        else:

            smoothed_center = tuple(map(int, detected_center))

        previous_ball_center = last_ball_center
        last_ball_center = smoothed_center
        last_ball_box = selected_ball["box"]
        last_ball_confidence = selected_ball["confidence"]
        missed_ball_frames = 0

    elif last_ball_center is not None:

        missed_ball_frames += 1

    if last_ball_center is not None and missed_ball_frames <= BALL_MAX_MISSED_FRAMES:

        if missed_ball_frames > 0 and previous_ball_center is not None:

            velocity = (
                np.array(last_ball_center) -
                np.array(previous_ball_center)
            )

            ball_center = tuple(map(int,
                np.array(last_ball_center) + velocity * missed_ball_frames
            ))

        else:

            ball_center = last_ball_center

        if 0 <= ball_center[0] < width and 0 <= ball_center[1] < height:

            x1, y1, x2, y2 = last_ball_box
            box_width = x2 - x1
            box_height = y2 - y1

            x1 = ball_center[0] - box_width // 2
            y1 = ball_center[1] - box_height // 2
            x2 = x1 + box_width
            y2 = y1 + box_height

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                BALL_COLOR,
                2
            )

            label = (
                f"BALL {last_ball_confidence:.2f}"
                if missed_ball_frames == 0
                else "BALL PREDICTED"
            )

            cv2.putText(
                frame,
                label,
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                BALL_COLOR,
                2
            )

    for track in referee_tracks.values():
        track["matched"] = False

    possible_matches = []

    for referee_id, track in referee_tracks.items():

        if track["missed"] >= REFEREE_MAX_MISSED_FRAMES:
            continue

        predicted = predict_position(track)

        velocity = np.linalg.norm(
            np.array(track["center"]) - np.array(track["previous"])
        )

        max_distance = REFEREE_BASE_MAX_DISTANCE + velocity * (
            track["missed"] + 1
        )

        for candidate_index, candidate in enumerate(referee_candidates):

            distance = np.linalg.norm(
                np.array(candidate["center"]) - predicted
            )

            if distance <= max_distance:
                possible_matches.append(
                    (distance, referee_id, candidate_index)
                )

    possible_matches.sort(key=lambda item: item[0])

    used_tracks = set()
    used_candidates = set()

    for distance, referee_id, candidate_index in possible_matches:

        if referee_id in used_tracks or candidate_index in used_candidates:
            continue

        track = referee_tracks[referee_id]
        candidate = referee_candidates[candidate_index]

        detected_center = np.array(candidate["center"])

        smoothed_center = (
            REFEREE_SMOOTHING * detected_center +
            (1 - REFEREE_SMOOTHING) * np.array(track["center"])
        )

        track["previous"] = track["center"]
        track["center"] = tuple(map(int, smoothed_center))
        track["box"] = candidate["box"]
        track["confidence"] = candidate["confidence"]
        track["missed"] = 0
        track["matched"] = True

        used_tracks.add(referee_id)
        used_candidates.add(candidate_index)

    for candidate_index, candidate in enumerate(referee_candidates):

        if candidate_index in used_candidates:
            continue

        referee_tracks[next_referee_id] = {
            "center": candidate["center"],
            "previous": candidate["center"],
            "box": candidate["box"],
            "confidence": candidate["confidence"],
            "missed": 0,
            "matched": True
        }

        next_referee_id += 1

    for referee_id in list(referee_tracks.keys()):

        track = referee_tracks[referee_id]

        if not track["matched"]:
            track["missed"] += 1

        if track["missed"] > REFEREE_MAX_MISSED_FRAMES:
            del referee_tracks[referee_id]
            continue

        if track["missed"] > 0:

            predicted = predict_position(track)
            referee_center = tuple(map(int, predicted))

        else:

            referee_center = track["center"]

        if not (
            0 <= referee_center[0] < width and
            0 <= referee_center[1] < height
        ):
            continue

        x1, y1, x2, y2 = track["box"]

        box_width = x2 - x1
        box_height = y2 - y1

        x1 = referee_center[0] - box_width // 2
        y1 = referee_center[1] - box_height // 2
        x2 = x1 + box_width
        y2 = y1 + box_height

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            REFEREE_COLOR,
            2
        )

        label = (
            f"REFEREE {track['confidence']:.2f}"
            if track["missed"] == 0
            else "REFEREE PREDICTED"
        )

        cv2.putText(
            frame,
            label,
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            REFEREE_COLOR,
            2
        )

    out.write(frame)

cap.release()
out.release()
cv2.destroyAllWindows()

print(f"Video saved to: {target}")


100%|██████████| 750/750 [01:02<00:00, 11.99it/s]

Video saved to: ./videos/result_video.mp4
